# load summary reports

In [11]:
from __future__ import annotations

import json
import re
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs pandas. Select the project .venv kernel, "
        "or install dependencies with: uv sync"
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")



# Set to None to load every model folder under logs/.
# Examples:
# MODEL_FILTER = "qwen3.6-plus"
# MODEL_FILTER = ["qwen3.6-plus", "gpt5.5"]
MODEL_FILTER = None

# Set True if you want CSV copies of the DataFrames written to analysis/outputs/.
EXPORT_CSV = False

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else Path(".").resolve()
LOGS_ROOT = PROJECT_ROOT / "logs"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "outputs"

assert LOGS_ROOT.exists(), f"Logs folder not found: {LOGS_ROOT}"


def normalise_model_filter(model_filter):
    if model_filter is None:
        return None
    if isinstance(model_filter, str):
        return {model_filter}
    return set(model_filter)


def discover_report_paths(logs_root: Path, model_filter=None) -> list[Path]:
    allowed_models = normalise_model_filter(model_filter)
    report_paths = []

    for model_dir in sorted(p for p in logs_root.iterdir() if p.is_dir()):
        if allowed_models is not None and model_dir.name not in allowed_models:
            continue

        for run_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
            report_path = run_dir / "task_run_report.json"
            if report_path.exists():
                report_paths.append(report_path)

    return report_paths


report_paths = discover_report_paths(LOGS_ROOT, MODEL_FILTER)
print(f"Found {len(report_paths)} reports")
for path in report_paths[:10]:
    print(path.relative_to(PROJECT_ROOT))


def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def leading_run_index(run_name: str):
    match = re.match(r"^(\d+)", run_name)
    return int(match.group(1)) if match else None


reports = []
for report_path in report_paths:
    run_dir = report_path.parent
    model_dir = run_dir.parent
    report = read_json(report_path)
    reports.append({
        "model_name": model_dir.name,
        "run_name": run_dir.name,
        "run_index": leading_run_index(run_dir.name),
        "report_path": str(report_path.relative_to(PROJECT_ROOT)),
        "report": report,
    })

len(reports)


report_rows = []
task_rows = []
tool_score_rows = []

for item in reports:
    report = item["report"]
    summary = report.get("summary", {}) or {}
    base = {
        "model_name": item["model_name"],
        "run_name": item["run_name"],
        "run_index": item["run_index"],
        "report_path": item["report_path"],
        "generated_at": report.get("generated_at"),
        "source_csv": report.get("source_csv"),
    }

    report_rows.append({**base, **summary})

    for result in report.get("results", []) or []:
        validation = result.get("ground_truth_validation") or {}
        query_trace = result.get("query_trace") or {}
        tool_appropriateness = result.get("tool_appropriateness") or {}

        task_base = {
            **base,
            "task_id": result.get("task_id"),
            "prompt": result.get("prompt"),
            "status": result.get("status"),
            "answer_type": result.get("answer_type"),
            "passed": validation.get("passed", result.get("passed")),
            "ground_truth_score": validation.get("score"),
            "validation_strategy": validation.get("strategy"),
            "ground_truth_validation": validation,
            "parameter_schema_valid_rate": result.get("parameter_schema_valid_rate"),
            "constraint_compliance_rate": result.get("constraint_compliance_rate"),
            "tool_appropriateness": tool_appropriateness.get("average_score"),
            "tool_appropriateness_weighted": tool_appropriateness.get("weighted_score"),
            "total_tokens": result.get("total_tokens"),
            "input_tokens": query_trace.get("input_tokens"),
            "output_tokens": query_trace.get("output_tokens"),
            "total_steps": result.get("total_steps"),
            "total_tool_calls": result.get("total_tool_calls"),
            "latency": result.get("latency"),
            "end_to_end_latency": result.get("end_to_end_latency"),
            "attempt_count": result.get("attempt_count"),
            "tools_used": result.get("tools_used") or query_trace.get("tools_used"),
            "final_answer": query_trace.get("final_answer", result.get("final_answer")),
        }
        task_rows.append(task_base)

        for call_index, scored_call in enumerate(tool_appropriateness.get("scored_calls") or [], start=1):
            tool_score_rows.append({
                **base,
                "task_id": result.get("task_id"),
                "call_index": call_index,
                **scored_call,
            })

reports_df = pd.DataFrame(report_rows)
task_runs_df = pd.DataFrame(task_rows)
tool_scores_df = pd.DataFrame(tool_score_rows)

if not task_runs_df.empty:
    task_runs_df["passed"] = task_runs_df["passed"].astype("boolean")
    numeric_columns = [
        "ground_truth_score",
        "parameter_schema_valid_rate",
        "constraint_compliance_rate",
        "tool_appropriateness",
        "tool_appropriateness_weighted",
        "total_tokens",
        "input_tokens",
        "output_tokens",
        "total_steps",
        "total_tool_calls",
        "latency",
        "end_to_end_latency",
        "attempt_count",
    ]
    for col in numeric_columns:
        if col in task_runs_df.columns:
            task_runs_df[col] = pd.to_numeric(task_runs_df[col], errors="coerce")

if not tool_scores_df.empty and "score" in tool_scores_df.columns:
    tool_scores_df["score"] = pd.to_numeric(tool_scores_df["score"], errors="coerce")

print(f"reports_df: {reports_df.shape[0]} rows x {reports_df.shape[1]} columns")
print(f"task_runs_df: {task_runs_df.shape[0]} rows x {task_runs_df.shape[1]} columns")
print(f"tool_scores_df: {tool_scores_df.shape[0]} rows x {tool_scores_df.shape[1]} columns")








Found 18 reports
logs\claude-optus-4.6\1anthropic_claude-opus-4.6_20260512T084016Z\task_run_report.json
logs\claude-optus-4.6\2anthropic_claude-opus-4.6_20260516T090351Z\task_run_report.json
logs\claude-optus-4.6\3anthropic_claude-opus-4.6_20260512T082634Z\task_run_report.json
logs\claude-sonnet-4.6\1anthropic_claude-sonnet-4.6_20260512T072817Z\task_run_report.json
logs\claude-sonnet-4.6\2anthropic_claude-sonnet-4.6_20260512T073036Z\task_run_report.json
logs\claude-sonnet-4.6\3anthropic_claude-sonnet-4.6_20260512T073516Z\task_run_report.json
logs\gpt5.5\1openai_gpt-5.5_20260512T075029Z\task_run_report.json
logs\gpt5.5\2openai_gpt-5.5_20260512T075044Z\task_run_report.json
logs\gpt5.5\3openai_gpt-5.5_20260512T075850Z\task_run_report.json
logs\kimi-k2.6\1moonshotai_kimi-k2.6_20260512T065725Z\task_run_report.json
reports_df: 18 rows x 28 columns
task_runs_df: 630 rows x 28 columns
tool_scores_df: 3106 rows x 12 columns


## Task-Level Performance Exploration

This section excludes `llama4` because its runs are unstable, then ranks tasks by task completion rate (TCR). Lower TCR means more model-task failures across the remaining models/runs.

In [12]:
EXCLUDED_MODELS = {"llama4"}
WORST_N = 15

TASK_CATEGORY_LABELS = {
    "A": "Discovery & Dataset Overview",
    "B": "Filtered Counting & Lookup",
    "C": "Aggregation & Temporal Pattern Analysis",
    "D": "Comparative Analysis",
    "E": "Exploratory NLP Analysis",
}

task_runs_for_task_analysis = task_runs_df[~task_runs_df["model_name"].isin(EXCLUDED_MODELS)].copy()
task_runs_for_task_analysis["task_category"] = task_runs_for_task_analysis["task_id"].astype(str).str.extract(r"^([A-Za-z])", expand=False).str.upper()
task_runs_for_task_analysis["task_category_name"] = task_runs_for_task_analysis["task_category"].map(TASK_CATEGORY_LABELS).fillna("Unknown")

task_tcr = task_runs_for_task_analysis.groupby(
    ["task_id", "task_category", "task_category_name"],
    as_index=False,
).agg(
    prompt=("prompt", "first"),
    models=("model_name", "nunique"),
    runs=("run_name", "nunique"),
    task_runs=("task_id", "count"),
    successful_task_runs=("passed", "sum"),
    task_completion_rate=("passed", "mean"),
    average_latency=("latency", "mean"),
    average_tokens=("total_tokens", "mean"),
)

task_tcr["failed_task_runs"] = task_tcr["task_runs"] - task_tcr["successful_task_runs"]

worst_task_tcr = task_tcr.sort_values(
    ["task_completion_rate", "failed_task_runs", "task_id"],
    ascending=[True, False, True],
)

print(f"Excluded models: {sorted(EXCLUDED_MODELS)}")
print(f"Analysed models: {sorted(task_runs_for_task_analysis['model_name'].unique())}")
display(worst_task_tcr.head(WORST_N))

Excluded models: ['llama4']
Analysed models: ['claude-optus-4.6', 'claude-sonnet-4.6', 'gpt5.5', 'kimi-k2.6', 'qwen3.6-plus']


,task_id,task_category,task_category_name,prompt,models,runs,task_runs,successful_task_runs,task_completion_rate,average_latency,average_tokens,failed_task_runs
24,D4,D,Comparative Analysis,"Comparing the daily occurrences of the terms ""...",5,15,15,8,0.5333,154.2328,"73,230.2000",7
34,E7,E,Exploratory NLP Analysis,"For the Reddit collection, retrieve the NLP me...",5,15,15,9,0.6000,25.0813,"9,282.4667",6
14,C1,C,Aggregation & Temporal Pattern Analysis,"Find out when the Twitter collection started, ...",5,15,15,11,0.7333,36.8490,"17,506.9333",4
28,E1,E,Exploratory NLP Analysis,"Over the first week in Jan 2025, in the Bluesk...",5,15,15,11,0.7333,148.9729,"37,097.6000",4
33,E6,E,Exploratory NLP Analysis,Compare the topic clusters found in the BlueSk...,5,15,15,11,0.7333,45.9549,"26,075.8000",4
12,B6,B,Filtered Counting & Lookup,"For BlueSky from 2025-02-01 to 2025-02-03, ret...",5,15,15,12,0.8000,13.1749,"12,150.3571",3
22,D2,D,Comparative Analysis,Compare the monthly post volume of Reddit and ...,5,15,15,12,0.8000,56.1546,"20,161.0000",3
23,D3,D,Comparative Analysis,which collection had the highest number of pos...,5,15,15,12,0.8000,98.1139,"52,868.8000",3
27,D7,D,Comparative Analysis,Compare Reddit activity between January and Se...,5,15,15,12,0.8000,42.3219,"17,498.0667",3
13,B7,B,Filtered Counting & Lookup,"In 2020, which available collection had the hi...",5,15,15,13,0.8667,48.4043,"23,033.4667",2


### Worst Tasks by Model

The pivot below shows each model's TCR for the worst tasks. This helps distinguish globally hard tasks from tasks that are mostly caused by one model.

In [13]:
worst_task_ids = worst_task_tcr.head(WORST_N)["task_id"].tolist()

task_model_tcr = task_runs_for_task_analysis.groupby(
    ["task_id", "model_name"],
    as_index=False,
).agg(task_completion_rate=("passed", "mean"))

task_model_tcr_matrix = task_model_tcr[task_model_tcr["task_id"].isin(worst_task_ids)].pivot_table(
    index="task_id",
    columns="model_name",
    values="task_completion_rate",
)

task_model_tcr_matrix = task_model_tcr_matrix.reindex(worst_task_ids)
task_model_tcr_matrix_percent = task_model_tcr_matrix.map(lambda value: f"{value:.2%}" if pd.notna(value) else "")
display(task_model_tcr_matrix_percent)

model_name,claude-optus-4.6,claude-sonnet-4.6,gpt5.5,kimi-k2.6,qwen3.6-plus
task_id,,,,,
D4,100.00%,33.33%,66.67%,0.00%,66.67%
E7,100.00%,66.67%,66.67%,33.33%,33.33%
C1,100.00%,66.67%,100.00%,100.00%,0.00%
E1,100.00%,100.00%,33.33%,100.00%,33.33%
E6,66.67%,100.00%,100.00%,33.33%,66.67%
B6,100.00%,100.00%,33.33%,66.67%,100.00%
D2,100.00%,33.33%,100.00%,66.67%,100.00%
D3,100.00%,100.00%,33.33%,66.67%,100.00%
D7,100.00%,66.67%,33.33%,100.00%,100.00%


## Ground Truth Validation by Task ID

Set a task id below to list every recorded `ground_truth_validation` payload for that task across all model runs.

In [15]:
TASK_ID_TO_INSPECT = "B7"

validation_rows = task_runs_df.loc[
    task_runs_df["task_id"].astype(str).str.upper() == str(TASK_ID_TO_INSPECT).upper(),
    [
        "task_id",
        "model_name",
        "run_name",
        "run_index",
        "status",
        "passed",
        "ground_truth_score",
        "validation_strategy",
        "ground_truth_validation",
        "report_path",
    ],
].copy()

if validation_rows.empty:
    available_task_ids = sorted(task_runs_df["task_id"].dropna().astype(str).unique().tolist())
    print(f"No rows found for task id {TASK_ID_TO_INSPECT!r}.")
    print("Available task ids:", ", ".join(available_task_ids))
else:
    validation_rows = validation_rows.sort_values(["model_name", "run_index", "run_name"]).reset_index(drop=True)
    validation_rows["ground_truth_validation"] = validation_rows["ground_truth_validation"].map(
        lambda value: json.dumps(value, indent=2, ensure_ascii=True) if isinstance(value, dict) else value
    )
    with pd.option_context("display.max_colwidth", None):
        display(validation_rows)


,task_id,model_name,run_name,run_index,status,passed,ground_truth_score,validation_strategy,ground_truth_validation,report_path
0,B7,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T084016Z,1,completed,True,1,scalar_deterministic,"{\n ""score"": 1,\n ""passed"": true,\n ""strategy"": ""scalar_deterministic"",\n ""answer_type"": ""scalar"",\n ""normalized_answer_type"": ""scalar"",\n ""dispatch_type"": ""scalar"",\n ""diagnostics"": {\n ""ground_truth_parse_error"": null,\n ""final_answer_parse_error"": null,\n ""ground_truth_parser_used"": ""json"",\n ""final_answer_parser_used"": ""json_fragment"",\n ""expected"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""actual"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""tolerance"": {\n ""abs"": 0.5,\n ""rel"": 1e-06\n },\n ""value_match"": true,\n ""unit_match"": true,\n ""abs_error"": 0.0,\n ""rel_error"": 0.0\n }\n}",logs\claude-optus-4.6\1anthropic_claude-opus-4.6_20260512T084016Z\task_run_report.json
1,B7,claude-optus-4.6,2anthropic_claude-opus-4.6_20260516T090351Z,2,completed,True,1,scalar_deterministic,"{\n ""score"": 1,\n ""passed"": true,\n ""strategy"": ""scalar_deterministic"",\n ""answer_type"": ""scalar"",\n ""normalized_answer_type"": ""scalar"",\n ""dispatch_type"": ""scalar"",\n ""diagnostics"": {\n ""ground_truth_parse_error"": null,\n ""final_answer_parse_error"": null,\n ""ground_truth_parser_used"": ""json"",\n ""final_answer_parser_used"": ""json_fragment"",\n ""expected"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""actual"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""tolerance"": {\n ""abs"": 0.5,\n ""rel"": 1e-06\n },\n ""value_match"": true,\n ""unit_match"": true,\n ""abs_error"": 0.0,\n ""rel_error"": 0.0\n }\n}",logs\claude-optus-4.6\2anthropic_claude-opus-4.6_20260516T090351Z\task_run_report.json
2,B7,claude-optus-4.6,3anthropic_claude-opus-4.6_20260512T082634Z,3,completed,True,1,scalar_deterministic,"{\n ""score"": 1,\n ""passed"": true,\n ""strategy"": ""scalar_deterministic"",\n ""answer_type"": ""scalar"",\n ""normalized_answer_type"": ""scalar"",\n ""dispatch_type"": ""scalar"",\n ""diagnostics"": {\n ""ground_truth_parse_error"": null,\n ""final_answer_parse_error"": null,\n ""ground_truth_parser_used"": ""json"",\n ""final_answer_parser_used"": ""json_fragment"",\n ""expected"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""actual"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""tolerance"": {\n ""abs"": 0.5,\n ""rel"": 1e-06\n },\n ""value_match"": true,\n ""unit_match"": true,\n ""abs_error"": 0.0,\n ""rel_error"": 0.0\n }\n}",logs\claude-optus-4.6\3anthropic_claude-opus-4.6_20260512T082634Z\task_run_report.json
3,B7,claude-sonnet-4.6,1anthropic_claude-sonnet-4.6_20260512T072817Z,1,completed,True,1,scalar_deterministic,"{\n ""score"": 1,\n ""passed"": true,\n ""strategy"": ""scalar_deterministic"",\n ""answer_type"": ""scalar"",\n ""normalized_answer_type"": ""scalar"",\n ""dispatch_type"": ""scalar"",\n ""diagnostics"": {\n ""ground_truth_parse_error"": null,\n ""final_answer_parse_error"": null,\n ""ground_truth_parser_used"": ""json"",\n ""final_answer_parser_used"": ""json_fragment"",\n ""expected"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""actual"": {\n ""value"": 28845.0,\n ""unit"": ""posts""\n },\n ""tolerance"": {\n ""abs"": 0.5,\n ""rel"": 1e-06\n },\n ""value_match"": true,\n ""unit_match"": true,\n ""abs_error"": 0.0,\n ""rel_error"": 0.0\n }\n}",logs\claude-sonnet-4.6\1anthropic_claude-sonnet-4.6_20260512T072817Z\task_run_report.json
4,B7,claude-sonnet-4.6,2anthropic_claude-sonnet-4.6_20260512T073036Z,2,completed,True,1,scalar_deterministic,"{\n ""score"": 1,\n ""passed"": true,\n ""strategy"": ""scalar_deterministic"",\n ""answer_type"": ""scalar"",\n ""normalized_answer_type"": ""scalar"",\n ""dispatch_type"": ""scalar"",\n ""diagnostics"": {\n ""ground_truth_parse_error"": null,\n ""final_answer_parse_error"": null,\n ""ground_truth_parser_used"": ""jso

### Tool Call Timestamps for `gdelt`

This section inspects each run's `tool_calls/<task_id>.jsonl` file and lists the timestamps for tool calls tied to `gdelt`.

In [19]:
tool_timestamp_rows = []

task_filter = task_runs_df["task_id"].astype(str).str.upper() == str(TASK_ID_TO_INSPECT).upper()
task_run_rows = task_runs_df.loc[task_filter, ["task_id", "model_name", "run_name", "run_index", "report_path"]].copy()
task_run_rows = task_run_rows.drop_duplicates().sort_values(["model_name", "run_index", "run_name"]).reset_index(drop=True)

for row in task_run_rows.itertuples(index=False):
    report_path = PROJECT_ROOT / str(row.report_path)
    tool_calls_path = report_path.parent / "tool_calls" / f"{row.task_id}.jsonl"

    if not tool_calls_path.exists():
        tool_timestamp_rows.append({
            "task_id": row.task_id,
            "model_name": row.model_name,
            "run_name": row.run_name,
            "run_index": row.run_index,
            "tool_calls_path": str(tool_calls_path.relative_to(PROJECT_ROOT)),
            "tool_name": None,
            "step_index": None,
            "timestamp": None,
            "match_source": "missing_tool_calls_file",
        })
        continue

    with tool_calls_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            payload = json.loads(line)

            parsed_arguments = payload.get("parsed_arguments")
            raw_arguments = payload.get("raw_arguments")

            parsed_match = isinstance(parsed_arguments, dict) and any(
                isinstance(value, str) and "gdelt" in value.lower()
                for value in parsed_arguments.values()
            )
            raw_match = isinstance(raw_arguments, str) and "gdelt" in raw_arguments.lower()

            if not (parsed_match or raw_match):
                continue

            tool_timestamp_rows.append({
                "task_id": row.task_id,
                "model_name": row.model_name,
                "run_name": row.run_name,
                "run_index": row.run_index,
                "tool_calls_path": str(tool_calls_path.relative_to(PROJECT_ROOT)),
                "tool_name": payload.get("tool_name"),
                "step_index": payload.get("step_index"),
                "timestamp": payload.get("timestamp"),
                "match_source": "parsed_arguments" if parsed_match else "raw_arguments",
            })

tool_timestamps_df = pd.DataFrame(tool_timestamp_rows)

if tool_timestamps_df.empty:
    print(f"No gdelt tool-call timestamps found for task id {TASK_ID_TO_INSPECT!r}.")
else:
    sort_columns = [column for column in ["model_name", "run_index", "run_name", "step_index", "timestamp"] if column in tool_timestamps_df.columns]
    tool_timestamps_df = tool_timestamps_df.sort_values(sort_columns, na_position="last").reset_index(drop=True)
    with pd.option_context("display.max_colwidth", None):
        display(tool_timestamps_df)


,task_id,model_name,run_name,run_index,tool_calls_path,tool_name,step_index,timestamp,match_source
0,B7,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T084016Z,1,logs\claude-optus-4.6\1anthropic_claude-opus-4.6_20260512T084016Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T08:41:14Z,parsed_arguments
1,B7,claude-optus-4.6,2anthropic_claude-opus-4.6_20260516T090351Z,2,logs\claude-optus-4.6\2anthropic_claude-opus-4.6_20260516T090351Z\tool_calls\B7.jsonl,text_search,2,2026-05-16T09:04:59Z,parsed_arguments
2,B7,claude-optus-4.6,3anthropic_claude-opus-4.6_20260512T082634Z,3,logs\claude-optus-4.6\3anthropic_claude-opus-4.6_20260512T082634Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T08:27:43Z,parsed_arguments
3,B7,claude-sonnet-4.6,1anthropic_claude-sonnet-4.6_20260512T072817Z,1,logs\claude-sonnet-4.6\1anthropic_claude-sonnet-4.6_20260512T072817Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T07:29:15Z,parsed_arguments
4,B7,claude-sonnet-4.6,2anthropic_claude-sonnet-4.6_20260512T073036Z,2,logs\claude-sonnet-4.6\2anthropic_claude-sonnet-4.6_20260512T073036Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T07:31:33Z,parsed_arguments
5,B7,claude-sonnet-4.6,3anthropic_claude-sonnet-4.6_20260512T073516Z,3,logs\claude-sonnet-4.6\3anthropic_claude-sonnet-4.6_20260512T073516Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T07:36:18Z,parsed_arguments
6,B7,gpt5.5,1openai_gpt-5.5_20260512T075029Z,1,logs\gpt5.5\1openai_gpt-5.5_20260512T075029Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T07:51:33Z,parsed_arguments
7,B7,gpt5.5,2openai_gpt-5.5_20260512T075044Z,2,logs\gpt5.5\2openai_gpt-5.5_20260512T075044Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T07:51:52Z,parsed_arguments
8,B7,gpt5.5,3openai_gpt-5.5_20260512T075850Z,3,logs\gpt5.5\3openai_gpt-5.5_20260512T075850Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T07:59:57Z,parsed_arguments
9,B7,kimi-k2.6,1moonshotai_kimi-k2.6_20260512T065725Z,1,logs\kimi-k2.6\1moonshotai_kimi-k2.6_20260512T065725Z\tool_calls\B7.jsonl,text_search,2,2026-05-12T06:59:45Z,parsed_arguments
